# Protocol lưu trữ / tham chiếu lịch sử
Notebook này giữ cấu hình cũ để truy xuất thí nghiệm. Lượt GĐ2 mới tập trung FedAvg dùng **kaggle_stage2_fedavg_v4.ipynb** với split content-aware v4. Không trộn trực tiếp accuracy của hai protocol để tính chênh lệch GĐ1–GĐ2.


# FedAvg tương thích GĐ1 — reference only
Chỉ train FedAvg alpha100/1 seed42, 5 client, 10 round. Centralized/Local-only là baseline lịch sử; khác quy tắc checkpoint và chưa chứng minh toàn bộ dữ liệu lịch sử tương đương. Không phải nghiệm thu nghiên cứu117job/3seed.

Nhập chính xác đường dẫn package ZIP và dataset được cấp. Không tự dò dataset gần giống. Notebook gọi CLI của package; không chứa trainer. Mỗi lệnh GPU có launcher giới hạn phiên8h, ngừng việc mới15phút trước cuối phiên, checkpoint theo round và ledger xuyên phiên. Không hứa vừa quota trước calibration GPU thật.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, zipfile, time
os.environ['MPLBACKEND'] = 'Agg'
# Kaggle T4 hosts expose two GPUs; this protocol uses only the first.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
PACKAGE_ZIP = Path('/kaggle/input/REPLACE_RELEASE/stage1_compat_hardened.zip')
DATASET_ROOT = Path('/kaggle/input/REPLACE_DATASET/raw/color')
PACKAGE_ROOT = Path('/kaggle/working/gd2_federated_learning')
OUTPUT_ROOT = Path('/kaggle/working/stage1_compat_output')
RESTORE_SOURCE = None  # e.g. Path('/kaggle/input/previous-output/stage1_compat_output')
QUOTA_REMAINING_HOURS = None  # Actual Kaggle GPU quota displayed now, not assumed 30
ALREADY_USED_HOURS = None     # Total GPU hours used in this 30h experiment, including other workflows/setup
# Do not use this notebook to reset quota accounting after changing accounts/cycles.
assert PACKAGE_ZIP.is_file(), 'Set exact release ZIP'
assert DATASET_ROOT.is_dir(), 'Set exact dataset root'
assert QUOTA_REMAINING_HOURS is not None and ALREADY_USED_HOURS is not None, 'Enter both actual usage numbers'
with zipfile.ZipFile(PACKAGE_ZIP) as archive:
    destination = PACKAGE_ROOT.parent.resolve()
    for name in archive.namelist():
        assert (destination / name).resolve().is_relative_to(destination), 'Unsafe archive path'
    if PACKAGE_ROOT.exists():
        for name in archive.namelist():
            if name.endswith('/'): continue
            target = destination / name
            assert target.is_file() and target.read_bytes() == archive.read(name), 'Installed package differs; use a clean extraction directory'
    else:
        archive.extractall(destination)
if RESTORE_SOURCE is not None:
    source = Path(RESTORE_SOURCE)
    assert (source / 'account_budget.v2.json').is_file(), 'Restore entire output including account ledger'
    assert not OUTPUT_ROOT.exists(), 'Refuse merge/overwrite existing output; choose clean output root'
    shutil.copytree(source, OUTPUT_ROOT)
os.chdir(PACKAGE_ROOT)


## Runtime và GPU
Dùng runtime torch2.6/torchvision0.21 đã kiểm thử. Khi cặp có sẵn không phù hợp, tạo venv không ensurepip rồi bootstrap pip bằng interpreter notebook. Thời gian setup GPU vẫn tiêu quota: cập nhật lại hai số trước calibration/run. Không tự nâng cả môi trường Kaggle.


In [ ]:
setup_start = time.monotonic()
PYTHON = sys.executable
pair = subprocess.run([PYTHON, '-c', "import torch,torchvision; assert torch.__version__.split('+')[0].startswith('2.6.'); assert torchvision.__version__.split('+')[0].startswith('0.21.'); assert torch.cuda.is_available(); assert torch.cuda.device_count()==1"], capture_output=True)
if pair.returncode:
    ENV_ROOT = Path('/tmp/stage1-compat-runtime')
    PYTHON = str(ENV_ROOT / 'bin/python')
    if not Path(PYTHON).exists():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(ENV_ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', 'pip==25.0.1'], check=True)
    subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(PACKAGE_ROOT / 'requirements-stage1.txt')], check=True)
subprocess.run([PYTHON, '-c', "import torch,torchvision; assert torch.cuda.is_available() and torch.cuda.device_count()==1; print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))"], check=True)
SETUP_HOURS = (time.monotonic() - setup_start) / 3600
print('Setup elapsed GPU-session hours:', SETUP_HOURS, 'Update usage/quota before launch, including idle/preflight time.')
def command(action, *extra):
    args = [PYTHON, '-m', 'stage1_compat', action, '--output-dir', str(OUTPUT_ROOT)]
    if action != 'collect': args += ['--data-dir', str(DATASET_ROOT)]
    return subprocess.run(args + list(extra), check=True)


## Preflight và dry-run
Kiểm tra hash toàn bộ ảnh so với inventory hiện tại đã pin, split/partition/histogram và nguồn GĐ1. Việc này có thể mất thời gian trên ổ đĩa chậm; GPU session vẫn tính giờ. Manifest v2 bất biến theo nội dung, không phụ thuộc đường dẫn restore.


In [ ]:
command('preflight')
command('dry-run')


## Calibration và khóa ngân sách
Điền lại quota/usage hiện tại ngay trước chạy. Calibration qua cả hai alpha, mỗi alpha đủ5client; không đánh giá test. Report khóa source/data/device/init,10round và hệ số1.5. Nếu không đủ ngân sách, dừng main và giữ output. Phiên sau giữ report này; không giảm round.


In [ ]:
QUOTA_REMAINING_HOURS = float(input('Kaggle quota GPU còn lại thực tế (giờ): '))
ALREADY_USED_HOURS = float(input('Tổng giờ GPU đã dùng trong thí nghiệm30h (kể cả setup/idle/phiên trước): '))
command('calibrate', '--quota-hours', str(QUOTA_REMAINING_HOURS), '--already-used-hours', str(ALREADY_USED_HOURS), '--device', 'cuda')
report = json.loads((OUTPUT_ROOT / 'calibration/calibration_report.json').read_text())
print({k: report[k] for k in ('status', 'safe_round_seconds', 'forecast_total_seconds', 'quota_sufficient')})


## Run hoặc resume
Nếu thời gian thực tế chậm hơn dự toán, runner pause thay vì đổi protocol. Muốn tiếp tục sau phiên, lưu output, attach và restore toàn bộ. Không chạy hai notebook dùng cùng output.


In [ ]:
QUOTA_REMAINING_HOURS = float(input('Quota GPU còn lại thực tế ngay trước run (giờ): '))
ALREADY_USED_HOURS = float(input('Tổng giờ GPU đã dùng đến lúc này trong thí nghiệm30h: '))
try:
    command('run', '--quota-hours', str(QUOTA_REMAINING_HOURS), '--already-used-hours', str(ALREADY_USED_HOURS), '--device', 'cuda')
finally:
    command('collect')


## Collect riêng trên CPU
Cell này không gọi training; sau restore chỉ cần package và toàn bộ output. CLI collect không cần dataset ảnh hoặc GPU. Dữ liệu thiếu/lỗi không được zero-fill. Xem collection_status.json, không suy luận completed từ sự tồn tại file best_model.pth.


In [ ]:
command('collect')
print((OUTPUT_ROOT / 'reports/collection_status.json').read_text())


## Lưu và phục hồi
Chạy cell archive, sau đó dùng Save Version và giữ output trên Kaggle; tải ZIP hoặc attach dataset output của phiên đã lưu vào notebook phiên mới. Xác minh ZIP có account_budget.v2.json, stage1_ledger.json, calibration, manifest v2 và toàn bộ checkpoints/metrics. /kaggle/working không tự bền sau phiên. Nếu session bị hard timeout trước archive, vẫn lưu/download toàn bộ thư mục output bằng Kaggle Output.

Ở phiên mới: đặt RESTORE_SOURCE trỏ thư mục đã attach, OUTPUT_ROOT trỏ vị trí chưa tồn tại, giữ nguyên package/version; chạy lại preflight, nhập quota/usage thực tế rồi run. Không xóa ledger để tăng quota. Artifact legacy của bản Antigravity trước hardening sẽ bị từ chối; giữ nguyên để kiểm tra, chưa có migration tự động.


In [ ]:
assert (OUTPUT_ROOT / 'account_budget.v2.json').is_file(), 'No supervised GPU command recorded yet'
archive = shutil.make_archive('/kaggle/working/stage1_compat_output_backup', 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name)
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('Save/download entire output:', archive)
